# UrbanWindow 3D, Phoenix: how much green do people see from their windows?
### I-GUIDE Summer School 2026
**Team 3: Seeing Green from Indoors**
- **Lead:** Debayan Mandal
- **Members:** Jiang Zheng, Segun Ojo, Lixi Xu, Yangchengsi Zhang

People spend most of their time indoors, yet we measure urban greenness from the **top down** (satellite NDVI). What a person sees out a window is different: **horizontal**, eye-level, and it **changes with height**. UrbanWindow 3D estimates this **window-view green ratio** for every floor of every building. We place an idealized observer at each floor along each wall and ask what it would see.

**Origin and Houston Study:**
Zongrong Li, GEAR Lab, Texas A&M (advisor Dr. Lei Zou). *Seeing Green from Indoors in 3D* (https://papers.ssrn.com/sol3/papers.cfm?abstract_id=6766522).

## Setup

In [ ]:
import sys, os, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))          
import uw_kit as uw
import pandas as pd, matplotlib.pyplot as plt
from PIL import Image

In [ ]:
ROOT = Path.cwd()
DATA_DIR = ROOT / "data"

In [ ]:
# If removal is needed
#import shutil
#shutil.rmtree(ROOT / "data", ignore_errors=True)
#shutil.rmtree(ROOT / "data_citywide", ignore_errors=True)

In [ ]:
# One-time download of the representative data set
UW_DATA_URL = "https://www.dropbox.com/scl/fi/jzb6905auo0qkbvzao1wl/uw_phoenix_data.tar.gz?rlkey=3a0uikafvabs0u1c2uxyzvgpc&st=zcpt8kj2&dl=1"
uw.ensure_dataset(ROOT, UW_DATA_URL)
#DATA_DIR = ROOT / "data"
print("DATA_DIR:", DATA_DIR)

## Rendering the Phoenix City

Phoenix has 524,647 buildings in total (Global Building Atlas). Rendering every one is far beyond the compute budget, so we render a stratified sample of 52,784: we keep every high-rise and take a 10 percent sample of low-rise and mid-rise buildings, stratified by ZIP code and NDVI quartile so the sample still represents every neighborhood and every greenness level.

| Rise class | Number | Sampled | Rate | Rule |
|---|---|---|---|---|
| Low-rise | 519,059 | 52,012 | 10.0 % | 10 percent |
| Mid-rise | 5,471 | 655 | 12.0 % | 10 percent |
| High-rise | 117 | 117 | 100.0 % | 100 percent |
| **Total** | **524,647** | **52,784** | **10.1 %** | |

As a team of four we render these 52,784 buildings straight from Google's Photorealistic 3D Tiles on Anvil. The city is split into four slices; each person renders one quarter (about 13,000 buildings) as a batch job with their own Cesium token, and the four slices pool into one shared folder to rebuild the full dataset.

### Get onto Anvil

Go to the I-GUIDE Platform (platform.i-guide.io) -> **Anvil HPC**, or directly to OnDemand at ondemand.anvil.rcac.purdue.edu, and log in with your **ACCESS** account.
Open a terminal from **Clusters -> Anvil Shell Access**

### Get Cesium token

The renderer streams **Google's 3D Tiles** through **Cesium ion**, which needs an access token. Make your own free one so the four tokens share the city's Cesium load.
Sign up at cesium.com/ion, open **Access Tokens**, and copy your **default** token. 

### Render, Pool and Assemble


I have staged the container, the render scripts, and the four building-footprint slices (`phoenix_geojson/part_1..to..4`) in the shared project space `/anvil/projects/x-ees260113/team3/`. Export your token and submit the shared batch script with your **slice name** *(as in part_1, part_2, etc.)* as the argument:

```
export APPTAINERENV_CESIUM_ION_TOKEN=paste-your-own-token-here
cd $SCRATCH
sbatch /anvil/projects/x-ees260113/team3/render_slice.sbatch part_1
```

Check on it at any time:
```
squeue -u $USER
```

The ~13,000 buildings render on a compute node (about 17 hours) into `phoenix_pool/` (`PD` = pending, `R` = running, `CG` = completing); you can log out and the job keeps going. Every job writes into the same `phoenix_pool/`, so the four slices pool automatically. 

When all four are in, one person assembles them into the notebook's tables in a single GPU job (never run it on login node):

```
sbatch /anvil/projects/x-ees260113/team3/assemble.sbatch
squeue -u $USER
```

Once the four slices are rendered and assembled, zip the two full-city tables (`per_viewpoint.parquet` and `phoenix_building_green_heat.gpkg`) and share them on a **public** Dropbox link with `dl=1`. Running the cell below overlays them onto the data loaded in Setup, so everything below re-runs on the full city. Leave it unrun to stay on the representative sample.

In [ ]:
FULLCITY_URL = "https://www.dropbox.com/scl/fi/rzbupqq6o6nddsbodw3ok/test_data_citywide.zip?rlkey=ob2atwurk5dhi6orq3jacjkdn&dl=1"   # Replace
LOAD_FULLCITY = False    # set True after render is assembled
if LOAD_FULLCITY:
    uw.load_fullcity(ROOT, FULLCITY_URL)

### Data Dictionary

**City Agnostic**: Requires building-footprint gpkg with heights and NDVI, land-surface-temperature raster, and social-vulnerability tracts. Point `make_geojson.py` and `assemble.py` at those, and the same notebook should run on any city.

In [ ]:
PV     = DATA_DIR / "per_viewpoint.parquet"
GIS    = DATA_DIR / "phoenix_building_green_heat.gpkg"
MODEL  = DATA_DIR / "model" / "best_model_phoenix.pth"
LABELS = DATA_DIR / "model" / "label_colors.json"
FRAMES = DATA_DIR / "frames_sample"
print("Data stored under:", DATA_DIR)

* `data/per_viewpoint.parquet` : One row per rendered window. Green ratio, floor, orientation, location. This is produced by the render.
* `data/phoenix_building_green_heat.gpkg` : This is per building. Green, NDVI, LST, SVI, spatial clusters. This is produced by assemble.
* `data/model/best_model_phoenix.pth` : The desert adapted segmentation model.
* `data_citywide/` : Whole-city version of the Parquet and GeoPackage files.
* `houston_buildings_enriched.csv` : Houston per-building table, for the Houston comparison.

## Methodology

### Step 1: Footprint to Viewpoint

We have only a footprint and a height.
Given these parameters, we decide where an observer stands and which way they look.
Floors are set to 3m. Every observer is 10m apart. The first one is set to 5m, so no one lands on the corner. Eye-height is set to 1.5m above the floor.
Our example here is of a 40m * 20m footprint, which is 30m tall.

In [ ]:
levels, vp = uw.viewpoints_for_box(height=30)
print(f"A 30 m building has {levels} floors, {len(vp)} viewpoints")
uw.plot_viewpoints(vp); plt.show()

## Step 2: Render and Feed to Segmentation

This renders a 900 * 900 photo, then labels every pixel with **DeepLabV3**. The desert-adapted model is used. First, one frame:

In [ ]:
model  = uw.load_segmenter(MODEL)
labels = json.loads(Path(LABELS).read_text())
img = Image.open(sorted(FRAMES.glob("*.jpg"))[0]).convert("RGB")
mask, ratios = uw.segment_image(img, model, labels)
fig, ax = plt.subplots(1, 2, figsize=(11, 5))
ax[0].imshow(img);  ax[0].axis("off"); ax[0].set_title("rendered window view")
ax[1].imshow(mask); ax[1].axis("off"); ax[1].set_title("segmentation")
plt.show()
print("green ratio (Vegetation):", round(ratios["Vegetation"], 3))

Now we point the segmenter at a folder of rendered frames and get a per-frame ratios table. This is also how the Anvil part works. Each row's green column is that window's green ratio.

In [ ]:
ratios_df = uw.segment_folder(FRAMES, model, labels, limit=12)
print(ratios_df.shape[0], "frames -> green ratios")
ratios_df[["filename", "green", "Building", "Sky", "Road", "Terrain"]].round(3).head()

## Result: We get the resultant tables (per-viewpoint and per-building)

Here we will use a sample where we deliberately chose three Phoenix neighborhoods that span the **urban-form gradient**:

| Neighborhood | ZCTA  | Urban form                                             |
|--------------|-------|--------------------------------------------------------|
| Downtown     | 85004 | High-rise, tall towers, dense, little ground green     |
| Tempe / ASU  | 85281 | Mid-rise, mixed heights, campus and apartments         |
| Arcadia      | 85016 | Suburban, low-rise homes, mature yard trees            |

In [ ]:
pv = uw.load_viewpoints(PV)
g  = uw.load_buildings(GIS)
print(f"{len(pv):,} viewpoints, {pv.osm_id.nunique():,} buildings; per-building layer: {len(g)} rows")
pv[["zcta", "rise_class", "floor", "height_m", "green", "ndvi"]].head()

### Question 1: Does a building that looks green from a satellite also look green from it's windows?

In [ ]:
pb = uw.per_building_green_ndvi(pv)
print(f"per-building green vs NDVI: r = {pb.green.corr(pb.ndvi):.2f}  (n={len(pb)})")
uw.plot_green_vs_ndvi(pb); plt.show()

* **r = 0.51** is a **moderate positive correlation**. Greener-from-above places do tend to have somewhat greener window views.
* **r-squared=0.26**, so NDVI explains about a quarter of the variation in window-green. The other 74% cannot be accounted for in the top-down view.

*NDVI is good at capturing neighborhood, not the building.*
Note: SAVI maybe worth checking too.

### Question 2: Would you expect higher floors to see more green or less? Does it depend on the building type?

In [ ]:
uw.plot_green_by_floor(pv); plt.show()

* **Low-Rise:** Greenery is quite decent at the street and then barely rises. Suburbanites don't need height to see green.
* **Mid-Rise:** Rises a crest at height 2-3. Presumably in level with the tree canopies. Most mid-rise viewpoints are clustered in that bin.
* **High-Rise:** It is very low at the street. A Downtown tower's ground view is mostly walls, roads and other buildings. It rises steadily with height as the view rises over the built-clutter.

*NDVI gives one number per location. Here, we can observe the vertical pattern more minutely, even within one building itself.*

**Notes:** 
* The 2-3 is missing from the high-rise due to a per-building viewpoint cap. It is a budget constraint set for Cesium usage. We didn't sample it.
* For our sample majority of the high-rises came from Downtown area. Mid-rises are more prevalent in Tempe area.

Viewpoint-counts:
|         | Downtown 85004 |	Arcadia 85016 |	Tempe 85281 |
|---------|----------------|------------------|-------------|
|High-rise|	414 | 180 | 168 |
|Mid-rise|	1,759 |	2,727 |	4,935 |
|Low-rise|  2,355 |	2,614 |	2,156 |

### Floor height vs. temperature and building form

Extending the floor analysis with surface temperature, and how it relates to window-view green across building forms.

In [ ]:
uw.plot_lst_by_floor(pv, g)

plt.show()

In [ ]:
uw.plot_lst_vs_green_by_floor(pv, g)

plt.show()

In [ ]:
uw.plot_lst_vs_green_by_building_form(pv, g)

plt.show()

### Question 3: Do greener buildings sit on cooler surfaces?

In [ ]:
for c in ["green", "ndvi", "svi"]:
    print(f"  {c:5s} vs LST:  r = {g[c].corr(g.lst):+.2f}")
uw.heat_divergence(g)

**Terms:**
* LST: *Land Surface Temperature* from Landsat in Celsius.
* green: *Window-view green ratio*
* ndvi: *Normalized Difference Vegetation Index*
* svi: *CDC Social Vulnerability Index*
* r: Pearson Correlation
* hidden green: NDVI says bare, but window says green
* exposed: NDVI says green, but window says bare.

**Result:**
* **green vs LST = −0.39** : Greener buildings sit on cooler surfaces.
* **ndvi vs LST = −0.35** : Satellite-green is also cooler, but slightly weaker than window-green. Very small margin however (0.04).
* **svi vs LST = +0.51** : More vulnerable areas are hotter.

| Group | mean LST | n |
|---|---|---|
| hidden_green | 54.58°C | 91 |
| typical | 55.83°C | 600 |
| exposed |  55.67°C | 98 |

* **Hidden Green buildings are cooler by ~1.1°C than exposed buildings.** So when the two measures conflict, the window-view goes with the cooler surface. The caveat, however, is the disagreements are in smaller numbers and gaps.

### Question 4: Is greenery or heat, evenly distributed?

Each building inherit's its tract's CDC SVI.

In [ ]:
print("SVI vs green:", round(g.svi.corr(g.green), 2), "   SVI vs LST:", round(g.svi.corr(g.lst), 2))
uw.plot_equity_corner(g); plt.show()
uw.equity_split(g)

**Results:**
* **SVI vs green = −0.29** : More vulnerable buildings have less window-green.
* **SVI vs LST = +0.51** More vulnerable buildings sit on hotter surfaces.

| Group | mean window-green | mean surface temp | n |
|---|---|---|---|
| Most vulnerable (top third SVI) | 0.027 | 56.9°C | 670 |
| Least vulnerable (bottom third SVI) | 0.060 | 54.5°C | 608 |

So the most vulnerable buildings have less than half the window-green and are ~2.4°C hotter.

### Question 5: Are the patterns spatially clustered, and where do low-green and high-heat overlap?

In [ ]:
%pip install -q libpysal esda

In [ ]:
print("double-burden hotspots:", len(uw.double_burden(g)))
{k: (round(v.I, 2), round(v.p_sim, 3)) for k, v in uw.morans(g).items()}

**Results:**
* **green: ( Moran's I=0.42, p=0.005):** Window-green is moderately clustered in space. Statistically significant.
* **lst: ( Moran's I=0.80, p=0.005):** Surface heat is strongly clustered. Heat comes in large contiguous zones. Statistically significant.
* **svi: ( Moran's I=0.95, p=0.005):** This result is not useful. SVI is assigned per census tract, so every building in a tract shares one value, which forces near-total clustering by construction.
* The *warning* is because the neighborhoods aren't connected.

## Visualization

Let's map it.
Projection: EPSG:32612

In [ ]:
%pip install -q contextily

In [ ]:
uw.plot_maps(g); plt.show()

# City-Wide Analysis

Let's check if the same results are obtained when the analysis is done citywide.
Use the full-city render if done. Otherwise use the representative 12,000-building sample and try the comparisons again.

In [ ]:
CITYWIDE = DATA_DIR.parent / "data_citywide"     # 12k city-wide sample (the full 52k run drops in here later)
g_city = uw.load_buildings(CITYWIDE / "phoenix_building_green_heat.gpkg")
print(f"city-wide buildings: {len(g_city):,}   ({(g_city.rise_class == 'low_rise').mean():.0%} low-rise)")

The current representative dataset has 12k buildings. Majority of which is low-rise, depicting the city of Phoenix correctly.

## Question 1: NDVI vs Window-Heat

In [ ]:
import matplotlib.pyplot as plt

#function for building green
def per_building_green_ndvi_city(g_city):
    return (
        g_city.dropna(subset=["osm_id", "green", "ndvi"])
              .groupby("osm_id", as_index=False)
              .agg(
                  green=("green", "mean"),
                  ndvi=("ndvi", "mean")
              )
    )

#function to plot
def plot_green_vs_ndvi_city(pb, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 5))

    r = pb["green"].corr(pb["ndvi"])

    ax.scatter(
        pb["ndvi"],
        pb["green"],
        s=10,
        alpha=0.4
    )

    ax.set_xlabel("Top-down NDVI")
    ax.set_ylabel("Building green fraction")
    ax.set_title(f"City-wide green fraction vs NDVI (r = {r:.2f})")

    return ax

In [ ]:
# call the function here to plot the city wide green fraction
pb2 = per_building_green_ndvi_city(g_city)

print(
    f"City-wide per-building green vs NDVI: "
    f"r = {pb2.green.corr(pb2.ndvi):.2f} "
    f"(n={len(pb2):,})"
)

plot_green_vs_ndvi_city(pb2)
plt.show()

## Question 2: Window-Green trend with floor

In [ ]:
pv_city = uw.load_viewpoints(CITYWIDE / "per_viewpoint.parquet")
gn = g_city[["osm_id", "ndvi"]].copy()
gn["osm_id"] = gn["osm_id"].astype(str)
pv_city["osm_id"] = pv_city["osm_id"].astype(str)
pv_city = pv_city.merge(gn.drop_duplicates("osm_id"), on="osm_id", how="left")
print(f"City-wide viewpoints: {len(pv_city):,}   buildings: {pv_city.osm_id.nunique():,}")

In [ ]:
uw.plot_green_by_floor(pv_city, zctas=None)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
uw.plot_con_green_by_floor(
    pv_city,
    ax=ax,
    zctas=None,
    min_count=15,
    building_weighted=True
)
plt.show()

## Question 3: Urban Heat

In [ ]:
def prepare_citywide_heat_data(g_city):
    required = {"green", "ndvi", "svi", "lst"}
    missing = required.difference(g_city.columns)

    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    g = g_city.dropna(
        subset=["green", "ndvi", "svi", "lst"]
    ).copy()

    g["green_pct"] = g["green"].rank(pct=True)
    g["ndvi_pct"] = g["ndvi"].rank(pct=True)

    return g

def heat_divergence_city(g):
    hidden = g[
        (g["ndvi_pct"] < 0.33) &
        (g["green_pct"] > 0.66)
    ]

    exposed = g[
        (g["ndvi_pct"] > 0.66) &
        (g["green_pct"] < 0.33)
    ]

    typical = g[
        g["green_pct"].between(0.33, 0.66)
    ]

    return {
        "hidden_green": (hidden["lst"].mean(), len(hidden)),
        "typical": (typical["lst"].mean(), len(typical)),
        "exposed": (exposed["lst"].mean(), len(exposed))
    }

In [ ]:
# Call the function and print the result
g_heat_city = prepare_citywide_heat_data(g_city)

print("City-wide correlations with LST:")

for c in ["green", "ndvi", "svi"]:
    print(
        f"  {c:5s} vs LST: "
        f"r = {g_heat_city[c].corr(g_heat_city['lst']):+.2f}"
    )
    
# Call and print the heat divergence, and round it to 2 decimal point.
heat = heat_divergence_city(g_heat_city)

heat_table = (
    pd.DataFrame(heat, index=["Mean LST (°C)", "Count"])
      .T
      .rename_axis("Group")
      .reset_index()
)
heat_table["Mean LST (°C)"] = heat_table["Mean LST (°C)"].round(2)
heat_table["Group"] = (
    heat_table["Group"]
    .str.replace("_", " ")
    .str.title()
)

print(heat_table)

In [ ]:
uw.plot_lst_vs_green_by_building_form(pv_city, g_city)

plt.show()

*Response*: 
- We might say urban heat is not solely influenced by visible vegetation.
- In essence, vegetation alone does not fully explain urban heat patterns.
- Vegetation and urban heat have a more complicated relationship.
- Although vegetation aids in cooling, its effects may be outweighed by urban design, the makeup of the land cover, and socioeconomic circumstances.

## Question 4: Social Vulnerability

In [ ]:
# City-wide Q4: are green and heat distributed evenly across social vulnerability?
print("SVI vs green:", round(g_city.svi.corr(g_city.green), 2), "   SVI vs LST:", round(g_city.svi.corr(g_city.lst), 2))
print("(sample baselines: SVI~green -0.29, SVI~LST +0.51)")
uw.plot_equity_corner(g_city); plt.show()
uw.equity_split(g_city)

Sample baselines: SVI~green = -0.29, SVI~LST = +0.51 (most-vulnerable buildings had roughly half the window-green and were ~2.4 C hotter). Check whether the same double disadvantage holds across the whole city.

## Extending Q4 - social & economic dimension

Decomposes the single composite SVI into its social / economic components and tests which one drives the green / heat gap.

**What this section does**
1. **Adds the sub-indices** - spatially joins CDC SVI 2022 to recover the 4 theme percentiles (socioeconomic, household, minority, housing/transport) per building.
2. **Tract-level aggregation** - averages window-green to census tract, SVI's native resolution.
3. **Analysis** - scatter plots (building & tract level), an LMG / Shapley variance decomposition, and standardized regression (tract-level, and building-level with tract-clustered SE).
4. **Key findings**
   - Racial/ethnic minority (THEME3) is the largest contributor (~39% of explained variance); socioeconomic status (THEME1) second (~29%).
   - The link is weak per building (R²≈0.05) but ~3× stronger aggregated to tract (R²≈0.17) - building-to-building green noise is unrelated to SVI.
   - More vulnerable tracts have less window-green and higher surface heat.
   - THEME1 and THEME3 are collinear (r≈0.81), hard to separate; housing/transport (THEME4) is essentially unrelated.

### Extending Q4 - from one SVI number to its social components

The baked `svi` column is only the **composite** index. CDC/ATSDR's SVI bundles **15 census variables into 4 themes**, each reported as a percentile rank. Decomposing it shows *which* dimension of vulnerability tracks with window-view green and heat.

**The four themes (SVI 2022):**

| Theme | Captures |
|---|---|
| `RPL_THEME1` Socioeconomic status | below 150% poverty, unemployment, housing-cost burden, no high-school diploma, no health insurance |
| `RPL_THEME2` Household characteristics | aged ≥65, aged ≤17, disability, single-parent households, limited English |
| `RPL_THEME3` Racial & ethnic minority | share in racial / ethnic minority groups |
| `RPL_THEME4` Housing type & transportation | multi-unit & mobile homes, crowding, no vehicle, group quarters |

**About the index:**
- Values are **percentile ranks 0-1** (higher = more vulnerable); `RPL_THEMES` is the overall composite.
- **SVI 2022** was **released May 2024**, and is the **first edition also offered at ZIP Code Tabulation Area (ZCTA) level** (alongside tract and county).
- We use the **tract-level** file for Maricopa County (1,009 tracts).

**Linking each building to a tract:**
- Building footprints (GlobalBuildingAtlas) carry only a ZIP (`zcta5`) - **no tract id**.
- So each building point (`lon`, `lat`) is **spatially joined** into the tract polygon that contains it, inheriting that tract's theme percentiles.
- **Validated:** this join reproduces the pre-baked composite `svi` *exactly* (zero difference), so the sub-themes are trustworthy. ~14 buildings fall in suppressed / zero-population tracts → `NaN`.

> Requires `data/svi_2022_maricopa_tracts.gpkg` (from the Anvil `team3/` kit); not committed to the repo.

In [ ]:
# Spatial join: attach GEOID + RPL_THEME1..4 to each building
SVI_TRACTS = DATA_DIR / "svi_2022_maricopa_tracts.gpkg"
g_se = uw.attach_svi_themes(g_city, SVI_TRACTS)

**How SVI is attached & aggregated**
- *Per building:* each building point (`lon`, `lat`) is spatially joined into the census tract polygon that contains it (point-in-polygon); it inherits that tract's composite SVI and 4 theme percentiles. SVI is constant within a tract.
- *Per tract:* for tract-level scatters, maps, and regressions, window-green is averaged over the buildings in each tract (mean by `GEOID`); SVI / themes are already tract-constant, so their per-tract value is unchanged.

In [ ]:
# Correlation of each SVI measure with window-green and surface heat
uw.svi_theme_summary(g_se)

**1. Does vulnerability track with window-view green?**

*Building level* (12,000 buildings) - composite first, then the four themes side by side, each with a linear fit.

In [ ]:
uw.plot_svi_vs_green(g_se, "svi"); plt.show()

In [ ]:
uw.plot_svi_themes_vs_green(g_se); plt.show()

*Census-tract level* - window-green averaged per tract (~380 tracts). SVI is a tract-level measure, so this is the matched resolution: building-to-building noise is averaged out and the relationship comes through more clearly (composite r goes from about -0.16 to -0.30).

In [ ]:
g_tract = uw.to_tract(g_se)
uw.plot_svi_vs_green(g_tract, "svi"); plt.show()

In [ ]:
uw.plot_svi_themes_vs_green(g_tract); plt.show()

**Which theme contributes most?** Standardized regression of green on all four themes at once, plus an LMG / Shapley split of the explained variance (robust to the themes being correlated).

Read with three caveats:
- **R² is small** (~0.05 building, ~0.17 tract): SVI explains only a slice of window-green.
- **SVI is tract-level**, so we report a tract-aggregated fit and a building-level fit with tract-clustered standard errors side by side.
- **THEME1 and THEME3 are collinear** (r≈0.81), so their individual betas trade off.

In [ ]:
# LMG / Shapley: each theme's share of explained variance
uw.svi_relimp(g_se)

In [ ]:
# Standardized regression, tract level (~380 tracts)
uw.svi_regression(g_se, level="tract")

In [ ]:
# Standardized regression, building level, cluster-robust SE by tract
uw.svi_regression(g_se, level="building")

**2. Green vs. surface heat, colored by vulnerability** - the equity corner (bottom-right = low green, high heat, most vulnerable).

In [ ]:
uw.plot_equity_corner(g_se); plt.show()

**3. Mapping the four sub-themes** - per building (points).

In [ ]:
uw.plot_svi_theme_maps_pts(g_se); plt.show()

*Same, by census tract* (choropleth - SVI's native level).

In [ ]:
uw.plot_svi_theme_maps(SVI_TRACTS, geoids=g_se["GEOID"]); plt.show()

**Composite SVI and window-green** - per building (points).

In [ ]:
uw.plot_svi_green_maps_pts(g_se); plt.show()

*Same, by census tract.*

In [ ]:
uw.plot_tract_svi_green(SVI_TRACTS, g_se); plt.show()

# Comparative Analysis: Phoenix vs Houston

UrbanWindow began in **Houston**, a humid, greener city. We have Houston's per-building table (`green`, `lst`, `svi`). Compare the two cities on the relationships that survived the Phoenix city-wide test: green-vs-heat and vulnerability-vs-heat. 

In [ ]:
hou = uw.load_houston(DATA_DIR.parent / "houston_buildings_enriched.csv")
print(f"Houston buildings: {len(hou):,}")
uw.compare_cities(g_city, hou)

# (Optional, Anvil GPU) Adapt the model to a new city

The segmentation model was trained on **Houston**. On desert Phoenix it failed. On a held-out test set it scored **0.00 IoU on Terrain** (it had never seen bare desert) and **0.17 on Vegetation**. We fine-tuned it on 168 Phoenix frames with pseudo-label masks; the same held-out test then jumped to **Terrain 0.31, Vegetation 0.41, Road 0.16 -> 0.40, Building 0.69 -> 0.82, Sky 0.88 -> 0.99**. That is the desert adaptation.

Training is GPU work, so it runs on an **Anvil GPU session**, not this notebook. 
I have staged a `finetune_kit` zip folder (containing the trainer, the Houston base model, and the frame/mask pairs) in the shared project space and kept an extracted folder. 
In an Anvil terminal:
```
# 1. Grab a short GPU session:
sinteractive -A ees260113-gpu -p gpu --gres=gpu:1 -t 00:30:00

# 2. On the GPU node, fine-tune (try 1 epoch first, then 8):
cd /anvil/projects/x-ees260113/team3/finetune_kit
apptainer exec --nv --bind "$PWD" /anvil/projects/x-ees260113/team3/urbanwindow3d.sif \
    python finetune.py --frames trainset_frames --masks trainset_labels \
    --weights best_model_base.pth --out best_model_NewCity.pth --epochs 8
```

`finetune.py` reshapes the DeepLabV3 head to 8 classes, trains, reports per-class IoU before and after, and saves the new checkpoint under a distinct name. The shipped Phoenix trainset reproduces the desert adaptation exactly; swap in your own city's frame/mask pairs to adapt to its morphology.

## Credits
- **Method and Houston case study:** Zongrong Li, GEAR Lab, Texas A&M (advisor Dr. Lei Zou). *Seeing Green from Indoors in 3D* (https://papers.ssrn.com/sol3/papers.cfm?abstract_id=6766522). 
- **Data:** Google Photorealistic 3D Tiles (Cesium ion), GlobalBuildingAtlas, Sentinel-2 NDVI, Landsat LST, CDC/ATSDR SVI 2022, Census TIGER.